# Objectives

### English



- The objective of this notebook is to effectively expand the International Player Database by incorporating club competition data for the players identified in the previous expansion analysis.

- This notebook reuses the existing player vector pipeline to build club-level player histories, merge them with the current international player database, and generate an improved version of the player database for future roster matching.


### Español



- El objetivo de esta notebook es expandir efectivamente la International Player Database incorporando datos de competiciones de clubes para los jugadores identificados en el análisis de expansión anterior.

- Esta notebook reutiliza el pipeline existente de construcción de vectores de jugadores para generar historiales de clubes, combinarlos con la base internacional actual y producir una nueva versión mejorada de la base de jugadores para futuras iteraciones del Current Roster Matching.

# Configuration

## Imports

In [73]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

## Paths

In [74]:
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
STATSBOMB_DATA = RAW_DIR / "statsbomb" / "data"
EVENTS_DIR = STATSBOMB_DATA / "events"


## Utility Functions

In [75]:
def build_player_vector(events):

    # GENERO DF DE EVENTS
    player_events = []

    for event in events:

        if "player" not in event:
            continue

        player_events.append({
            "player_id": event["player"]["id"],
            "player_name": event["player"]["name"],
            "team_name": event["team"]["name"],
            "event_type": event["type"]["name"]
        })

    player_events_df = pd.DataFrame(player_events)

    # FILTRO EVENTOS ADMINISTRATIVOS
    ADMIN_EVENTS = [
        "Starting XI",
        "Half Start",
        "Half End",
        "Substitution",
        "Tactical Shift",
        "Injury Stoppage",
        "Referee Ball-Drop",
    ]

    player_events_clean = player_events_df[
        ~player_events_df["event_type"].isin(ADMIN_EVENTS)
    ].copy()

    # PLAYER VECTOR
    player_vector = (
        player_events_clean
        .pivot_table(
            index=[
                "player_id",
                "player_name",
                "team_name"
            ],
            columns="event_type",
            values="event_type",
            aggfunc="count",
            fill_value=0
        )
        .reset_index()
    )

    # TESTEO
    assert player_vector["player_id"].is_unique, "Hay jugadores duplicados"
    assert player_vector.isna().sum().sum() == 0, "Hay valores NaN"
    assert player_vector["team_name"].nunique() == 2, "No hay exactamente 2 equipos"
    assert "Pass" in player_vector.columns, "No existe la columna Pass"

    print("All tests passed")

    return player_vector

In [76]:
def load_events(match_id, events_dir):
    event_file = events_dir / f"{match_id}.json"

    with open(event_file, "r", encoding="utf-8") as f:
        events = json.load(f)

    return events

In [77]:
def build_player_database_from_matches(matches_df, events_dir, save_path=None):
    all_player_vectors = []

    for match_id in matches_df["match_id"].unique():
        events = load_events(match_id, events_dir)

        player_vector = build_player_vector(events)
        player_vector["match_id"] = match_id

        all_player_vectors.append(player_vector)

    player_vectors_match_level = pd.concat(all_player_vectors, ignore_index=True)

    numeric_cols = [
        col for col in player_vectors_match_level.select_dtypes(include="number").columns
        if col not in ["match_id", "player_id"]
    ]

    team_names = (
        player_vectors_match_level
        .groupby(["player_id", "player_name"])["team_name"]
        .apply(lambda x: " | ".join(sorted(x.unique())))
        .reset_index()
    )

    player_database = (
        player_vectors_match_level
        .groupby(["player_id", "player_name"])[numeric_cols]
        .sum()
        .reset_index()
    )

    player_database = player_database.merge(
        team_names,
        on=["player_id", "player_name"],
        how="left"
    )

    assert player_database["player_id"].is_unique
    assert player_database.isna().sum().sum() == 0
    assert "Pass" in player_database.columns

    print("Player database created successfully")
    print("Match-level shape:", player_vectors_match_level.shape)
    print("Player database shape:", player_database.shape)

    if save_path is not None:
        player_database.to_csv(save_path, index=False)
        print("Saved to:", save_path)

    return player_database, player_vectors_match_level

## Load Data

In [78]:
club_expansion_candidates = pd.read_csv(
    "../data/processed/club_player_expansion_candidates_v01.csv"
)

club_expansion_candidates.head()

,country,roster_name,position,shirt_number,found_in_club_data,club_player_id,club_player_name,club_team_name
0,Algeria (ALG),GOUIRI Amine,Forward,9,True,4435.0,Amine Gouiri,Rennes
1,Algeria (ALG),HADJAM Jaouen,Defender,13,True,45089.0,Jaouen Hadjam,Nantes
2,Argentina (ARG),RULLI Geronimo,Goalkeeper,12,True,6694.0,Gerónimo Rulli,Villarreal
3,Argentina (ARG),LOPEZ Jose,Forward,21,True,49302.0,José Manuel Cabrera López,Villarreal
4,Argentina (ARG),MEDINA Facundo,Defender,25,True,30415.0,Facundo Axel Medina,Lens


### Checks

In [79]:
club_expansion_candidates.columns.tolist()

['country',
 'roster_name',
 'position',
 'shirt_number',
 'found_in_club_data',
 'club_player_id',
 'club_player_name',
 'club_team_name']

In [80]:
club_expansion_candidates.head()

,country,roster_name,position,shirt_number,found_in_club_data,club_player_id,club_player_name,club_team_name
0,Algeria (ALG),GOUIRI Amine,Forward,9,True,4435.0,Amine Gouiri,Rennes
1,Algeria (ALG),HADJAM Jaouen,Defender,13,True,45089.0,Jaouen Hadjam,Nantes
2,Argentina (ARG),RULLI Geronimo,Goalkeeper,12,True,6694.0,Gerónimo Rulli,Villarreal
3,Argentina (ARG),LOPEZ Jose,Forward,21,True,49302.0,José Manuel Cabrera López,Villarreal
4,Argentina (ARG),MEDINA Facundo,Defender,25,True,30415.0,Facundo Axel Medina,Lens


## Load Club Lineups

In [81]:
lineups_dir = DATA_DIR / "raw" / "statsbomb" / "data" / "lineups"

lineup_rows = []

for lineup_file in lineups_dir.glob("*.json"):
    match_id = int(lineup_file.stem)

    with open(lineup_file, "r", encoding="utf-8") as f:
        lineups = json.load(f)

    for team in lineups:
        team_name = team["team_name"]

        for player in team["lineup"]:
            lineup_rows.append({
                "match_id": match_id,
                "team_name": team_name,
                "player_id": player["player_id"],
                "player_name": player["player_name"]
            })

lineups_df = pd.DataFrame(lineup_rows)

lineups_df.head()

,match_id,team_name,player_id,player_name
0,15946,Barcelona,3109,Malcom Filipe Silva de Oliveira
1,15946,Barcelona,3501,Philippe Coutinho Correia
2,15946,Barcelona,5203,Sergio Busquets i Burgos
3,15946,Barcelona,5211,Jordi Alba Ramos
4,15946,Barcelona,5213,Gerard Piqué Bernabéu


#### Check

In [82]:
print("Lineup rows:", lineups_df.shape[0])
print("Unique matches:", lineups_df["match_id"].nunique())
print("Unique players:", lineups_df["player_id"].nunique())

Lineup rows: 161958
Unique matches: 4235
Unique players: 11889


# Build Candidate Lineups DF

In [83]:
candidate_player_ids = (
    club_expansion_candidates["club_player_id"]
    .dropna()
    .astype(int)
    .unique()
)

candidate_lineups = lineups_df[
    lineups_df["player_id"].isin(candidate_player_ids)
].copy()

In [84]:
print("Candidate players:", candidate_lineups["player_id"].nunique())
print("Candidate matches:", candidate_lineups["match_id"].nunique())

Candidate players: 56
Candidate matches: 371


In [85]:
candidate_matches_df = (
    candidate_lineups[["match_id"]]
    .drop_duplicates()
    .sort_values("match_id")
    .reset_index(drop=True)
)

candidate_matches_df.head()

,match_id
0,7533
1,7550
2,7568
3,7586
4,9575


#### Checks

In [86]:
print("Matches to process:", candidate_matches_df.shape[0])

Matches to process: 371


In [87]:
missing_candidate_ids = set(candidate_player_ids) - set(candidate_lineups["player_id"])

club_expansion_candidates[
    club_expansion_candidates["club_player_id"].astype(int).isin(missing_candidate_ids)
]

,country,roster_name,position,shirt_number,found_in_club_data,club_player_id,club_player_name,club_team_name


In [88]:
print("Candidate rows:", club_expansion_candidates.shape[0])
print("Unique roster players:", club_expansion_candidates["roster_name"].nunique())
print("Unique club player IDs:", club_expansion_candidates["club_player_id"].nunique())
print("Missing club player IDs:", club_expansion_candidates["club_player_id"].isna().sum())
print("Club player IDs found in lineups:", candidate_lineups["player_id"].nunique())

Candidate rows: 57
Unique roster players: 57
Unique club player IDs: 56
Missing club player IDs: 0
Club player IDs found in lineups: 56


One duplicated `club_player_id` was detected among the 57 candidate rows. Therefore, the expansion process will operate over 56 unique club player IDs.

# Build Club Player Database

In [89]:
club_player_database, club_match_player_vectors = (
    build_player_database_from_matches(
        candidate_matches_df,
        EVENTS_DIR
    )
)

All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests passed
All tests pass

In [90]:
print("Club player database:", club_player_database.shape)
print("Match-level vectors:", club_match_player_vectors.shape)

Club player database: (2892, 29)
Match-level vectors: (10815, 30)


In [91]:
team_names = (
    club_match_player_vectors
    .groupby(["player_id", "player_name"])["team_name"]
    .apply(lambda x: " | ".join(sorted(x.unique())))
    .reset_index()
)

numeric_cols = [
    col for col in club_match_player_vectors.select_dtypes(include="number").columns
    if col not in ["match_id", "player_id"]
]

club_player_database = (
    club_match_player_vectors
    .groupby(["player_id", "player_name"])[numeric_cols]
    .sum()
    .reset_index()
)

club_player_database = club_player_database.merge(
    team_names,
    on=["player_id", "player_name"],
    how="left"
)

In [92]:
assert club_player_database["player_id"].is_unique
assert club_player_database.isna().sum().sum() == 0
assert "Pass" in club_player_database.columns

print("All tests passed.")
print("Club player database:", club_player_database.shape)

All tests passed.
Club player database: (2892, 29)


# Merge Player Databases

In [93]:
international_player_database_v01 = pd.read_csv(
    PROCESSED_DIR / "international_player_database_v01.csv"
)

international_player_database_v01.head()

,player_id,player_name,team_name,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,...,Pass,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On
0,2941,Ismaïla Sarr,Senegal,11,2,3,3.0,0.0,378,31,...,213,1.0,1.0,183,0.0,20,0.0,1.0,0.0,0.0
1,2948,Nabil Fekir,France,6,1,1,0.0,0.0,59,9,...,40,0.0,0.0,32,0.0,4,0.0,0.0,0.0,0.0
2,2954,Youri Tielemans,Belgium,14,2,4,1.0,0.0,475,24,...,506,0.0,0.0,165,0.0,10,0.0,0.0,0.0,0.0
3,2956,Bertrand Isidore Traoré,Burkina Faso,4,1,1,0.0,0.0,98,7,...,69,1.0,1.0,13,0.0,11,0.0,0.0,0.0,0.0
4,2972,Marcus Thuram,France,10,2,3,1.0,0.0,193,14,...,104,0.0,0.0,73,1.0,14,0.0,0.0,0.0,0.0


In [94]:
print("International player database v01:", international_player_database_v01.shape)
print("Club player database v01:", club_player_database.shape)

International player database v01: (2279, 32)
Club player database v01: (2892, 29)


In [95]:
combined_player_database = pd.concat(
    [
        international_player_database_v01,
        club_player_database
    ],
    ignore_index=True
)

combined_player_database.shape

(5171, 33)

In [101]:
id_cols = [
    "player_id"
]

numeric_cols = [
    col for col in combined_player_database.select_dtypes(include="number").columns
    if col != "player_id"
]

team_names = (
    combined_player_database
    .groupby(id_cols)["team_name"]
    .apply(lambda x: " | ".join(sorted(x.dropna().unique())))
    .reset_index()
)

international_player_database_v02 = (
    combined_player_database
    .groupby(id_cols)[numeric_cols]
    .sum()
    .reset_index()
)

international_player_database_v02 = international_player_database_v02.merge(
    team_names,
    on=id_cols,
    how="left"
)

international_player_database_v02.head()

,player_id,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,Carry,...,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For,team_name
0,2935,0.0,0.0,0.0,3.0,0.0,399,26,16,344,...,2.0,107,4.0,5,0.0,0.0,0.0,0.0,0.0,Paris Saint-Germain
1,2936,0.0,0.0,0.0,0.0,0.0,12,3,2,13,...,0.0,18,0.0,0,0.0,0.0,0.0,0.0,0.0,Guingamp
2,2937,0.0,0.0,0.0,0.0,0.0,50,5,1,42,...,0.0,27,0.0,0,0.0,0.0,0.0,0.0,0.0,Montpellier
3,2940,0.0,0.0,0.0,0.0,0.0,9,1,0,9,...,0.0,4,0.0,0,0.0,0.0,0.0,0.0,0.0,Strasbourg
4,2941,11.0,2.0,3.0,5.0,0.0,547,39,8,384,...,2.0,250,0.0,30,0.0,1.0,0.0,0.0,0.0,Senegal


### Checks

In [102]:
print("International player database v01:", international_player_database_v01.shape)
print("Club player database v01:", club_player_database.shape)
print("Combined raw database:", combined_player_database.shape)
print("International player database v02:", international_player_database_v02.shape)

International player database v01: (2279, 32)
Club player database v01: (2892, 29)
Combined raw database: (5171, 33)
International player database v02: (4058, 32)


In [103]:
assert international_player_database_v02["player_id"].is_unique
assert international_player_database_v02.isna().sum().sum() == 0
assert "Pass" in international_player_database_v02.columns

print("All tests passed.")

All tests passed.


In [104]:
international_player_database_v02[
    international_player_database_v02["player_id"].duplicated(keep=False)
].sort_values("player_id")

,player_id,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,Carry,...,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For,team_name


In [99]:
players_before = international_player_database_v01["player_id"].nunique()
players_after = international_player_database_v02["player_id"].nunique()
players_added = players_after - players_before

print("Players before:", players_before)
print("Players after:", players_after)
print("Players added:", players_added)
print("Growth rate:", round(players_added / players_before * 100, 2), "%")

Players before: 2279
Players after: 4058
Players added: 1779
Growth rate: 78.06 %


# International Player Database V02

In [110]:
player_names = (
    combined_player_database
    .groupby("player_id")["player_name"]
    .first()
    .reset_index()
)

team_names = (
    combined_player_database
    .groupby("player_id")["team_name"]
    .apply(lambda x: " | ".join(sorted(x.dropna().unique())))
    .reset_index()
)

international_player_database_v02 = (
    combined_player_database
    .groupby("player_id")[numeric_cols]
    .sum()
    .reset_index()
)

international_player_database_v02 = (
    international_player_database_v02
    .merge(player_names, on="player_id")
    .merge(team_names, on="player_id")
)

In [111]:
international_player_database_v02 = international_player_database_v02[
    ["player_id", "player_name", "team_name"] +
    [col for col in international_player_database_v02.columns
     if col not in ["player_id", "player_name", "team_name"]]
]

#### Checks

In [112]:
assert international_player_database_v02["player_id"].is_unique
assert international_player_database_v02.isna().sum().sum() == 0
assert "Pass" in international_player_database_v02.columns

print("All tests passed.")
print("International player database v02:", international_player_database_v02.shape)

All tests passed.
International player database v02: (4058, 33)


# Save Results

In [113]:
club_player_database.to_csv(
    PROCESSED_DIR / "club_player_database_v01.csv",
    index=False
)

club_match_player_vectors.to_csv(
    PROCESSED_DIR / "club_match_player_vectors_v01.csv",
    index=False
)

international_player_database_v02.to_csv(
    PROCESSED_DIR / "international_player_database_v02.csv",
    index=False
)

print("Files saved successfully.")

Files saved successfully.


# Validation Summary

In [114]:
validation_summary = pd.DataFrame({
    "Metric": [
        "International Player Database v01",
        "Club Player Database",
        "International Player Database v02",
        "Players Added",
        "Growth Rate (%)"
    ],
    "Value": [
        international_player_database_v01["player_id"].nunique(),
        club_player_database["player_id"].nunique(),
        international_player_database_v02["player_id"].nunique(),
        players_added,
        round(players_added / players_before * 100, 2)
    ]
})

validation_summary

,Metric,Value
0,International Player Database v01,2279.00
1,Club Player Database,2892.00
2,International Player Database v02,4058.00
3,Players Added,1779.00
4,Growth Rate (%),78.06


# Conclusion

### English

- Successfully built a club player database from the selected expansion candidates.
- Reused the existing player vector pipeline through a generalized database construction function.
- Merged international and club player histories into a unified historical player database.
- Refined the data model by treating `player_id` as the unique player identifier while preserving player names and team history as descriptive attributes.
- Expanded the International Player Database from 2,279 to 4,058 unique players (+78.06%).
- The resulting database is ready for a new Current Roster Matching iteration using the expanded historical information.

### Español

- Se construyó exitosamente una base histórica de jugadores utilizando competiciones de clubes.
- Se reutilizó el pipeline existente de vectores de jugadores mediante una función generalizada para la construcción de bases de datos.
- Se integraron los historiales internacionales y de clubes en una única base histórica de jugadores.
- Se refinó el modelo de datos utilizando `player_id` como identificador único del jugador, preservando el nombre y el historial de equipos como atributos descriptivos.
- La International Player Database se expandió de 2.279 a 4.058 jugadores únicos (+78,06%).
- La nueva base quedó preparada para ejecutar una nueva iteración del Current Roster Matching con una cobertura histórica significativamente mayor.